# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed 
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Note: dataset.metadata is an object, not a dict. Use attributes to access metadata.
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print all record sets (using their `@id`), their field `@id`s, and columns for inspection.

In [ ]:
# List all record sets with their @id and fields

record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.id}: {field.name} (type: {field.data_type})")
        print("  Columns:")
        for column in rs.columns:
            print(f"    - {column.id}: {column.name}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we will use the first record set (if available).

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}.")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

# For the rest of the notebook, select the first record set (if available) as our working example
if record_set_ids:
    record_set_id = record_set_ids[0]
else:
    record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (if available) for demonstration.

In [ ]:
if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Find a numeric field
    numeric_fields = []
    record_set_md = [rs for rs in dataset.metadata.record_sets if rs.id == record_set_id][0]
    for field in record_set_md.fields:
        # Use types according to schema.org
        if field.data_type in ['Integer','Float','Number','schema:Integer','schema:Float','schema:Number']:
            # Prefer field ID over field name
            if field.id in df.columns:
                numeric_fields.append(field.id)
            elif field.name in df.columns:
                numeric_fields.append(field.name)
    print(f"Numeric fields found: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using {numeric_field} as a numeric field for analysis.")
        # Filter out missing/invalid values
        df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df_numeric.mean() if pd.notnull(df_numeric).any() else 10
        filtered_df = df[df_numeric > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (df_numeric - df_numeric.mean()) / df_numeric.std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a categorical/grouping field
        group_field = None
        for field in record_set_md.fields:
            if field.data_type in ['Text','schema:Text','String','schema:String'] and field.id != numeric_field:
                if field.id in df.columns:
                    group_field = field.id
                    break
                elif field.name in df.columns:
                    group_field = field.name
                    break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and record_set_id in dataframes and 'numeric_field' in locals():
    df = dataframes[record_set_id].copy()
    # Check for numeric data in the field (after conversion)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored metadata and structure of the dataset using `mlcroissant`.
- Identified available record sets and their respective fields and columns by `@id`.
- Loaded data into DataFrames for further processing and analysis.
- Demonstrated basic EDA, including filtering by a numeric field and normalizing values.
- Visualized the distribution of numeric values (where available).

This notebook can be extended for deeper analysis, modeling, and advanced visualizations depending on specific research questions.